# Atividade — IDHM por Unidade da Federação\n\nAnálise da evolução do IDHM usando **pandas** e **matplotlib**.

## 1. Abrindo o CSV\n\nO arquivo usa `;` como separador e vírgula como separador decimal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Tabela4.csv", sep=";", decimal=",")
df.head()


## 2. Limpando colunas vazias

In [ ]:
df = df.dropna(axis=1, how="all")
df.info()

## 3. Estados ordenados pelo maior IDH em 2024

In [ ]:
df_ordenado = df.sort_values("2024", ascending=False)
df_ordenado[["Sigla", "Estado", "2024"]]

## 4. Estado com maior melhora entre 1991 e 2024

In [ ]:
df["Melhora_1991_2024"] = df["2024"] - df["1991"]

maior_melhora = df.loc[df["Melhora_1991_2024"].idxmax()]
print(f"Estado: {maior_melhora['Estado']} ({maior_melhora['Sigla']})")
print(f"Melhora: {maior_melhora['Melhora_1991_2024']:.3f}")

## 5. Existe algum estado em que o IDH piorou?\n\nPrimeiro verificamos 1991 → 2024. Depois verificamos quedas em intervalos intermediários.

In [ ]:
piorou_no_periodo = df[df["Melhora_1991_2024"] < 0][["Sigla", "Estado", "Melhora_1991_2024"]]

if piorou_no_periodo.empty:
    print("Nenhum estado teve IDH menor em 2024 do que em 1991.")
else:
    print(piorou_no_periodo)

In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
quedas = []
for _, linha in df.iterrows():
    for ano_anterior, ano_atual in zip(anos[:-1], anos[1:]):
        if linha[ano_atual] < linha[ano_anterior]:
            quedas.append({"Sigla": linha["Sigla"], "Estado": linha["Estado"], "De": ano_anterior, "Para": ano_atual, "IDH_anterior": linha[ano_anterior], "IDH_atual": linha[ano_atual]})
pd.DataFrame(quedas)

## 6. Formato longo com `melt`

In [ ]:
anos = [c for c in df.columns if str(c).isdigit()]
print(anos)

id_vars = [c for c in df.columns if c not in anos]
df_longo = df.melt(id_vars=id_vars, value_vars=anos, var_name="Ano", value_name="IDH")
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo.head()

## 7. Plotando apenas Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=1.8)
ax.set_title("Evolução do IDH — Minas Gerais")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 8. Evolução do IDH de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()

## Respostas\n- Maior IDH em 2024: Distrito Federal (0,866).\n- Maior melhora entre 1991 e 2024: Tocantins.\n- Piora entre 1991 e 2024: nenhum estado.\n- Existem quedas em alguns intervalos intermediários, identificadas no código.